In [ ]:
import pandas as pd

#Loading the dataset and checking if data is in the same folder
df_matches = pd.read_csv('results.csv')

#Converting date column to datetime objects
df_matches['date'] = pd.to_datetime(df_matches['date'])

#checking data
print("Dataset Shape:", df_matches.shape)
display(df_matches.head())

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import seaborn

print("Python executable:", sys.executable)
print("Working directory:", os.getcwd())
print("Files in folder:", os.listdir())
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Scikit-learn version:", sklearn.__version__)

In [ ]:
#Dropping any rows where the match didn't happen or scores are missing
df_matches = df_matches.dropna(subset=['home_score', 'away_score']).copy()

#Filtering for the modern era that is post the year 2000
modern_era_cutoff = '2000-01-01'
df_clean = df_matches[df_matches['date'] >= modern_era_cutoff].copy()

#Checking the type of tournaments
print("Total matches after 2000:", df_clean.shape[0])
print("\nTop 10 Tournament Types:")
print(df_clean['tournament'].value_counts().head(10))

In [ ]:
import pandas as pd
import numpy as np

#Splitting the data into Home and Away perspectives
df_home = df_clean[['date', 'home_team', 'home_score', 'away_score']].copy()
df_home = df_home.rename(columns={'home_team': 'team', 'home_score': 'goals_for', 'away_score': 'goals_against'})

df_away = df_clean[['date', 'away_team', 'away_score', 'home_score']].copy()
df_away = df_away.rename(columns={'away_team': 'team', 'away_score': 'goals_for', 'home_score': 'goals_against'})
df_team_matches = pd.concat([df_home, df_away]).sort_values(by=['team', 'date']).reset_index(drop=True)

#Calculating Rolling Form (Average goals scored/conceded in the previous 5 matches) using shift() to ensure we don't include current match result
window_size = 5

df_team_matches['form_goals_for'] = df_team_matches.groupby('team')['goals_for'].transform(
    lambda x: x.shift().rolling(window=window_size, min_periods=1).mean()
)

df_team_matches['form_goals_against'] = df_team_matches.groupby('team')['goals_against'].transform(
    lambda x: x.shift().rolling(window=window_size, min_periods=1).mean()
)

#Filling any early NA values (teams playing their first few matches) with 0
df_team_matches = df_team_matches.fillna(0)

#Sanity check
print("Argentina's Form Check:")
display(df_team_matches[df_team_matches['team'] == 'Argentina'].tail(10))

In [ ]:
#Isolating just the form columns we want to bring over
df_form_only = df_team_matches[['date', 'team', 'form_goals_for', 'form_goals_against']]

#Merging form data for the Home Team
df_model = pd.merge(
    df_clean,
    df_form_only,
    left_on=['date', 'home_team'],
    right_on=['date', 'team'],
    how='left'
).rename(columns={
    'form_goals_for': 'home_form_goals_for',
    'form_goals_against': 'home_form_goals_against'
}).drop('team', axis=1)

#Merging form data for the Away Team
df_model = pd.merge(
    df_model,
    df_form_only,
    left_on=['date', 'away_team'],
    right_on=['date', 'team'],
    how='left'
).rename(columns={
    'form_goals_for': 'away_form_goals_for',
    'form_goals_against': 'away_form_goals_against'
}).drop('team', axis=1)

#Creating the Target Variable (y) for our classification model
#Encoding - 2 = Home Win, 1 = Draw, 0 = Away Win
conditions = [
    (df_model['home_score'] > df_model['away_score']),
    (df_model['home_score'] == df_model['away_score']),
    (df_model['home_score'] < df_model['away_score'])
]
choices = [2, 1, 0]
df_model['target'] = np.select(conditions, choices, default=np.nan)

#Cleaing up any NaNs that might have slipped through the merge
df_model = df_model.dropna()

print("Final Feature Matrix Shape:", df_model.shape)
display(df_model[['date', 'home_team', 'away_team', 'home_form_goals_for', 'away_form_goals_for', 'target']].tail())

In [ ]:
import numpy as np
import pandas as pd

#Applying weights and neutral flags to df_clean
def get_tournament_weight(tournament_name):
    if 'FIFA World Cup' in tournament_name and 'qualification' not in tournament_name:
        return 1.0
    elif 'Confederations Cup' in tournament_name or 'Copa America' in tournament_name or 'Euro' in tournament_name and 'qualification' not in tournament_name:
        return 0.8
    elif 'qualification' in tournament_name:
        return 0.6
    elif 'Nations League' in tournament_name:
        return 0.5
    elif 'Friendly' in tournament_name:
        return 0.25
    else:
        return 0.4

df_clean['match_weight'] = df_clean['tournament'].apply(get_tournament_weight)
df_clean['is_neutral'] = df_clean['neutral'].astype(int)

#Re running the Step 4 Merge so df_model inherits the new columns
df_model = pd.merge(
    df_clean, df_form_only,
    left_on=['date', 'home_team'], right_on=['date', 'team'], how='left'
).rename(columns={'form_goals_for': 'home_form_goals_for', 'form_goals_against': 'home_form_goals_against'}).drop('team', axis=1)

df_model = pd.merge(
    df_model, df_form_only,
    left_on=['date', 'away_team'], right_on=['date', 'team'], how='left'
).rename(columns={'form_goals_for': 'away_form_goals_for', 'form_goals_against': 'away_form_goals_against'}).drop('team', axis=1)

#Re applying the target variable
conditions = [
    (df_model['home_score'] > df_model['away_score']),
    (df_model['home_score'] == df_model['away_score']),
    (df_model['home_score'] < df_model['away_score'])
]
df_model['target'] = np.select(conditions, [2, 1, 0], default=np.nan)
df_model = df_model.dropna()

#Defining X and y for the SVM (This will now work perfectly!)
features = [
    'home_form_goals_for', 'home_form_goals_against',
    'away_form_goals_for', 'away_form_goals_against',
    'match_weight', 'is_neutral'
]
X = df_model[features]
y = df_model['target']

print("Success! The missing columns are injected.")
print("Shape of Feature Matrix (X):", X.shape)
print("Shape of Target Vector (y):", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# 1. Define our Feature Matrix (X) and Target Vector (y)
# We isolate only the predictive numeric columns, dropping metadata like dates and names
features = [
    'home_form_goals_for', 'home_form_goals_against',
    'away_form_goals_for', 'away_form_goals_against',
    'match_weight', 'is_neutral'
]

X = df_model[features]
y = df_model['target']

# 2. Split the pitch: 80% for training, 20% for testing our predictions
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Standardize the data so the SVM hyperplanes aren't distorted
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Initialize and train the model WITH balanced class weights
print("Retraining the SVM with balanced weights...")
svm_model_balanced = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
svm_model_balanced.fit(X_train_scaled, y_train)

# 5. Evaluate the newly balanced model
y_pred_balanced = svm_model_balanced.predict(X_test_scaled)
print("\n--- Balanced Model Evaluation ---")
print(classification_report(y_test, y_pred_balanced, target_names=['Away Win (0)', 'Draw (1)', 'Home Win (2)']))

In [ ]:
# 1. Define standard Elo mathematical functions
def get_expected_score(rating_a, rating_b):
    # Calculates the probability of Team A winning based on Elo difference
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def update_elo(rating, expected, actual, k=30):
    # k is the weight of the match.
    return rating + k * (actual - expected)

# 2. Initialize a dictionary to track everyone's live rating (Baseline is 1500)
current_elo = {}

home_elos = []
away_elos = []

# 3. Loop through history chronologically to calculate pre-match Elo for every game
print("Calculating historical Elo ratings...")
for index, row in df_clean.sort_values('date').iterrows():
    home = row['home_team']
    away = row['away_team']

    # If a team is new to the dataset, give them the baseline 1500 rating
    if home not in current_elo: current_elo[home] = 1500
    if away not in current_elo: current_elo[away] = 1500

    # Store the pre-match ratings to use as features later
    home_elos.append(current_elo[home])
    away_elos.append(current_elo[away])

    # Determine the actual outcome for the Elo math (1 = Home Win, 0.5 = Draw, 0 = Away Win)
    if row['home_score'] > row['away_score']:
        actual_home, actual_away = 1.0, 0.0
    elif row['home_score'] < row['away_score']:
        actual_home, actual_away = 0.0, 1.0
    else:
        actual_home, actual_away = 0.5, 0.5

    # Calculate expected outcomes
    expected_home = get_expected_score(current_elo[home], current_elo[away])
    expected_away = get_expected_score(current_elo[away], current_elo[home])

    # Update their live ratings using the match weight we created earlier
    k_adjusted = 30 * row['match_weight']
    current_elo[home] = update_elo(current_elo[home], expected_home, actual_home, k_adjusted)
    current_elo[away] = update_elo(current_elo[away], expected_away, actual_away, k_adjusted)

# 4. Attach these shiny new features to our clean dataset
df_clean_sorted = df_clean.sort_values('date').copy()
df_clean_sorted['home_elo'] = home_elos
df_clean_sorted['away_elo'] = away_elos

# Let's see who the top 5 teams are at the end of our dataset
print("\nTop 5 Teams by Final Elo Rating:")
top_teams = sorted(current_elo.items(), key=lambda x: x[1], reverse=True)[:5]
for team, rating in top_teams:
    print(f"{team}: {rating:.0f}")

In [ ]:
# 1. Merge the new Elo ratings into our existing model dataframe
df_model = pd.merge(
    df_model,
    df_clean_sorted[['date', 'home_team', 'away_team', 'home_elo', 'away_elo']],
    on=['date', 'home_team', 'away_team'],
    how='left'
)

# Clean up any potential NaNs from the merge
df_model = df_model.dropna()

# 2. Define our Upgraded Feature Matrix (X)
features_upgraded = [
    'home_elo', 'away_elo', # <-- The new heavy hitters
    'home_form_goals_for', 'home_form_goals_against',
    'away_form_goals_for', 'away_form_goals_against',
    'match_weight', 'is_neutral'
]

X_upgraded = df_model[features_upgraded]
y_upgraded = df_model['target']

# 3. Split and Scale the new pitch
X_train_up, X_test_up, y_train_up, y_test_up = train_test_split(X_upgraded, y_upgraded, test_size=0.2, random_state=42)

scaler_upgraded = StandardScaler()
X_train_scaled_up = scaler_upgraded.fit_transform(X_train_up)
X_test_scaled_up = scaler_upgraded.transform(X_test_up)

# 4. Retrain the SVM Engine
print("Retraining the SVM with Historical Elo Ratings...")
svm_model_final = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
svm_model_final.fit(X_train_scaled_up, y_train_up)

# 5. Evaluate the Ultimate Model
y_pred_final = svm_model_final.predict(X_test_scaled_up)
print("\n--- Final Model Evaluation (Elo + Form) ---")
print(classification_report(y_test_up, y_pred_final, target_names=['Away Win (0)', 'Draw (1)', 'Home Win (2)']))

In [ ]:
import numpy as np
from collections import Counter

# 1. The Match Prediction Engine
def get_match_probabilities(team_home, team_away):
    # In a real app, you would dynamically pull their live Elo and Form here.
    # For this simulation, we will construct a dummy feature vector representing a tight match.
    # Features: [home_elo, away_elo, home_gf, home_ga, away_gf, away_ga, weight, neutral]
    # Let's pretend they are evenly matched (Elo 1800) on neutral ground (1.0 weight, 1 neutral)
    dummy_features = [[1800, 1750, 2.0, 0.5, 1.8, 0.8, 1.0, 1]]

    # Scale and predict
    scaled_features = scaler_upgraded.transform(dummy_features)
    probs = svm_model_final.predict_proba(scaled_features)[0]

    # probs output is [P(Away Win), P(Draw), P(Home Win)]
    return probs[0], probs[1], probs[2]

# 2. The Single Match Simulator
def simulate_knockout_match(team_a, team_b):
    p_away, p_draw, p_home = get_match_probabilities(team_a, team_b)

    # In knockout football, there are no draws. If the SVM predicts a draw,
    # we simulate extra time/penalties by essentially tossing a coin (50/50).
    outcomes = [team_b, 'Draw', team_a]
    result = np.random.choice(outcomes, p=[p_away, p_draw, p_home])

    if result == 'Draw':
        return np.random.choice([team_a, team_b])
    return result

# 3. The Monte Carlo Bracket Simulator
def run_monte_carlo_tournament(iterations=10000):
    champions = []

    print(f"Running Monte Carlo Simulation ({iterations} universes)...")

    for _ in range(iterations):
        # Semi-Finals
        finalist_1 = simulate_knockout_match("Spain", "Brazil")
        finalist_2 = simulate_knockout_match("France", "Argentina")

        # The Final
        winner = simulate_knockout_match(finalist_1, finalist_2)
        champions.append(winner)

    return Counter(champions)

# 4. Execute the Simulation
results = run_monte_carlo_tournament(11000)

print("\n--- World Cup Monte Carlo Results (11000 Simulations) ---")
total = sum(results.values())
for team, wins in results.most_common():
    win_percentage = (wins / total) * 100
    print(f"{team}: {wins} tournament wins ({win_percentage:.2f}%)")

Chapter 1

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np

# 1. The Correlation Matrix
# We analyze X_upgraded to see how our engineered features interact
plt.figure(figsize=(10, 8))
correlation_matrix = X_upgraded.corr()

# Generate a clean heatmap using Seaborn
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Matrix: Pre-Match Variables")
plt.tight_layout()
plt.show()

# 2. Principal Component Analysis (PCA)
# We apply PCA to our SCALED training data to see the mathematical variance
pca = PCA()
pca.fit(X_train_scaled_up)

# Calculate the cumulative explained variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

# Plot the PCA curve
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--', color='b')
plt.axhline(y=0.90, color='r', linestyle='-') # The 90% variance threshold
plt.text(1.5, 0.91, '90% Variance Threshold', color='red')

plt.title("PCA: Cumulative Explained Variance")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Variance")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3. Print the hard numbers for your thesis text
print("--- PCA Breakdown ---")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"Principal Component {i+1}: {var*100:.2f}% of variance explained")

Chapter 2

In [ ]:
from sklearn.neural_network import MLPClassifier

# 1. Initialize the Neural Network Architecture
# We'll use two hidden layers (16 neurons, then 8 neurons) with ReLU activation.
# max_iter is set to 1000 to ensure the network has time to converge.
print("Training the Deep Learning Engine (MLP)...")
mlp_model = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=42
)

# 2. Fit the model to our scaled Elo + Form data
mlp_model.fit(X_train_scaled_up, y_train_up)

# 3. Evaluate the Neural Network
y_pred_mlp = mlp_model.predict(X_test_scaled_up)
print("\n--- Neural Network (MLP) Evaluation ---")
print(classification_report(y_test_up, y_pred_mlp, target_names=['Away Win (0)', 'Draw (1)', 'Home Win (2)']))

# 4. Compare the baseline accuracy of the two architectures
svm_acc = svm_model_final.score(X_test_scaled_up, y_test_up)
mlp_acc = mlp_model.score(X_test_scaled_up, y_test_up)

print(f"\nModel Showdown - SVM Accuracy: {svm_acc*100:.2f}% | MLP Accuracy: {mlp_acc*100:.2f}%")